In [ ]:
import sqlite3
import pandas as pd
import re
from google.colab import files

In [ ]:
# Kết nối và đọc dữ liệu
conn = sqlite3.connect("khoahoc_vn.db")
df = pd.read_sql_query("SELECT * FROM articles", conn)
conn.close()

In [ ]:
def clean_author_col(df):
  if 'author' in df.columns:
    df['author'] = (df['author']
    .str.replace('các tác giả', '', case=False, regex=False)
    .str.replace('*', '', regex=False)
    # thay xuống dòng bằng dấu phẩy
    .str.replace('\n', ',', regex=False)
    # đổi mọi ký tự không phải chữ ở đầu
    .str.replace(r'[^\wÀ-ỹ\s]+', ',', regex=True)
    # chuẩn hóa khoảng trắng quanh dấu phẩy
    .str.replace(r'\s*,\s*', ',', regex=True)
    # xử lý nhiều dấy phẩy liên tiếp
    .str.replace(r',+', ',', regex=True)
    # bỏ dấu phẩy đầu, cuối
    .str.strip(',')
    .str.strip())


  return df

In [ ]:
def clean_content_col(df, col="content"):
    if col not in df.columns:
        return df

    new_col = []

    for text in df[col]:
        if pd.isna(text):
            new_col.append("N/A")
            continue

        text = str(text)
        low = text.lower()

        pos = -1
        for kw in ["tóm tắt", "abstract", "summary"]:
            p = low.find(kw)
            if p != -1:
                pos = p
                break

        if pos != -1:
            text = text[pos:]

        # cắt phần tài liệu tham khảo
        text = re.split(r'tài liệu tham khảo', text, flags=re.IGNORECASE)[0]

        new_col.append(text)

    df[col] = new_col
    return df

In [ ]:
# xóa ký tự lạ (tránh lỗi IllegalCharacterError)
def clean_illegal_chars(val):
    if isinstance(val, str):
        return re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', val)
    return val

df = clean_author_col(df)
df = clean_content_col(df)
df = df.map(clean_illegal_chars)

In [ ]:
# xuất file csv
csv_file = "ds-1000b.csv"
df.to_csv(csv_file, index=False)

files.download(csv_file)

In [ ]:
# xuất file sql

def export_to_postgres_sql(db_name="khoahoc_vn.db", sql_file="data_postgres.sql"):
    # 1. Kết nối DB lấy dữ liệu từ SQLite
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    cursor.execute("SELECT title, author, abstract, content, url, category FROM articles")
    rows = cursor.fetchall()
    conn.close()

    # 2. Hàm dọn dẹp và escape ký tự đặc biệt
    def clean_and_escape(text):
        if text is None: return "NULL"
        # Xóa ký tự lạ gây lỗi hiển thị
        text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', str(text))
        # Escape dấu nháy đơn bằng cách gấp đôi dấu nháy (chuẩn SQL)
        return text.replace("'", "''")

    # 3. Ghi vào file SQL theo chuẩn PostgreSQL
    table_name = "articles"
    with open(sql_file, "w", encoding="utf-8") as f:
        # Lệnh khởi tạo cấu trúc bảng cho Postgres
        f.write(f"DROP TABLE IF EXISTS {table_name};\n\n")
        f.write(f"CREATE TABLE {table_name} (\n")
        # Trong Postgres dùng SERIAL để tự động tăng ID
        f.write("    id SERIAL PRIMARY KEY,\n")
        f.write("    title TEXT,\n")
        f.write("    author TEXT,\n")
        f.write("    abstract TEXT,\n")
        f.write("    content TEXT,\n")
        f.write("    url TEXT,\n")
        f.write("    category TEXT\n")
        f.write(");\n\n")

        for row in rows:
            title, author, abstract, content, url, category = row

            # Xóa cụm "các tác giả"
            clean_author = author.replace("các tác giả", "").strip() if author else "N/A"

            # Tạo câu lệnh INSERT
            sql = (f"INSERT INTO {table_name} (title, author, abstract, content, url, category) VALUES ("
                   f"'{clean_and_escape(title)}', "
                   f"'{clean_and_escape(clean_author)}', "
                   f"'{clean_and_escape(abstract)}', "
                   f"'{clean_and_escape(content)}', "
                   f"'{clean_and_escape(url)}', "
                   f"'{clean_and_escape(category)}');\n")
            f.write(sql)

    # 4. Tải file về
    files.download(sql_file)

export_to_postgres_sql()